# MYZ307E — Machine Learning for Electronics and Communications Engineering
## SNR-Adaptive Gating for Automatic Modulation Classification via Adaptive Wavelet Network

**Team Members:** Egemen Durmaz, Eren Dervişoğlu, İlker Doğan  
**Reference Paper:** Zhang et al., IEEE TCCN 2023, DOI: 10.1109/TCCN.2023.3252580  
**Original GitHub:** https://github.com/zjwfufu/AWN  
**Dataset:** RadioML2016.10a — DeepSig

---
### Notebook Structure
1. Environment Setup
2. Dataset Loading & Visualization
3. Baseline AWN Evaluation (Reproduction)
4. SAG Module — Architecture & Integration
5. SAG-AWN Training
6. Results & Comparison
7. Ablation Study
8. Save All Results to Drive

---
## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
!cp -r "/content/drive/MyDrive/MYZ307E_AWN_Project/AWN" /content/
!cp "/content/drive/MyDrive/MYZ307E_AWN_Project/data/RML2016.10a_dict.pkl" /content/AWN/data/
!cp "/content/drive/MyDrive/MYZ307E_AWN_Project/checkpoint/2016.10a_AWN.pkl" /content/AWN/checkpoint/

!pip install torch torchvision pyyaml matplotlib scikit-learn tqdm cairosvg pandas -q

os.chdir('/content/AWN')
print('✅ Environment ready!')

---
## 2. Dataset Loading & Visualization

**RadioML2016.10a:** 220,000 I/Q samples, 11 modulation classes, SNR: −20 to +18 dB, split 60/20/20.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

with open('./data/RML2016.10a_dict.pkl', 'rb') as f:
    data = pickle.load(f, encoding='bytes')

snrs = sorted(set([k[1] for k in data.keys()]))
mods = sorted(set([k[0] for k in data.keys()]))

print(f'Modulation classes ({len(mods)}): {[m.decode() for m in mods]}')
print(f'SNR range: {min(snrs)} dB to {max(snrs)} dB ({len(snrs)} levels)')
print(f'Total samples: {len(mods) * len(snrs) * 1000:,}')
print(f'Signal shape: 2 x 128 (I/Q samples)')

In [ ]:
# Visualize BPSK signals at different SNR levels
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mod = b'BPSK'

for ax, snr in zip(axes, [-20, 0, 18]):
    sample = data[(mod, snr)][0]
    ax.plot(sample[0], label='I channel', alpha=0.8)
    ax.plot(sample[1], label='Q channel', alpha=0.8)
    ax.set_title(f'BPSK at SNR = {snr} dB')
    ax.set_xlabel('Sample index')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('BPSK Signal at Different SNR Levels', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Baseline AWN Evaluation (Reproduction)

We reproduce AWN results using the pre-trained checkpoint from the original authors.

In [ ]:
# Run baseline AWN evaluation
!python main.py --mode eval --dataset 2016.10a

In [ ]:
# Display per-SNR accuracy curve
import cairosvg
from IPython.display import Image

cairosvg.svg2png(
    url='/content/AWN/inference/2016.10a_0/result/acc/acc.svg',
    write_to='/content/baseline_acc.png'
)
Image('/content/baseline_acc.png')

**Baseline AWN Reproduction Results:**

| Metric | Original Paper | Our Reproduction |
|--------|---------------|------------------|
| Overall Accuracy | ~62–63% | **64.32%** |
| Macro F1-Score | — | **66.14%** |
| Kappa Coefficient | — | **0.6048** |

---
## 4. SAG Module — Architecture & Integration

**Motivation:** AWN applies identical forward pass regardless of SNR, ignoring available signal quality information.

**Our contribution:** SNR-Adaptive Gating (SAG) module that dynamically weights wavelet sub-bands based on estimated SNR.

```
SNR (scalar) → Linear(1→32) → ReLU → Linear(32→N_subbands) → Sigmoid → gate
output = x + gate ⊙ x   (residual connection preserves original features)
```

SAG is inserted **after SE-attention layer, before final FC classifier** in AWN's forward pass.

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/AWN')

from models.model_sag import AWN_SAG

model = AWN_SAG(
    num_classes=11, num_levels=1, in_channels=64,
    kernel_size=3, latent_dim=320,
    regu_details=0.01, regu_approx=0.01
)

total_params = sum(p.numel() for p in model.parameters())
sag_params   = sum(p.numel() for n, p in model.named_parameters() if 'sag' in n)
awn_params   = total_params - sag_params

print(f'AWN parameters (pre-trained): {awn_params:,}')
print(f'SAG parameters (ours):        {sag_params:,}')
print(f'Total parameters:             {total_params:,} ({total_params/1e6:.2f}M)')
print(f'SAG overhead:                 {sag_params/total_params*100:.1f}% of total')

---
## 5. SAG-AWN Training

**Strategy:** Differential learning rates
- AWN backbone: lr = 1e-5 (preserve pre-trained features)
- SAG module: lr = 1e-3 (fast adaptation)

AWN checkpoint loaded → SAG trained on top.

In [ ]:
# Train SAG-AWN (skip if checkpoint already exists)
import os
if not os.path.exists('./training/2016.10a_2/models/2016.10a_AWN_SAG.pkl'):
    !python main_sag.py
else:
    print('✅ SAG-AWN checkpoint already exists, skipping training.')

---
## 6. Results & Comparison

In [1]:
# Generate all visuals: confusion matrix, per-SNR table, class-wise F1
!python generate_visuals.py

python3: can't open file '/content/generate_visuals.py': [Errno 2] No such file or directory


In [ ]:
# Display per-SNR comparison
from IPython.display import Image
Image('./visuals/per_snr_accuracy.png')

In [ ]:
# Display confusion matrix
Image('./visuals/confusion_matrix_sag.png')

In [ ]:
# Final results summary
import pandas as pd

results = pd.DataFrame({
    'Model':            ['CNN Baseline*', 'LSTM Baseline*', 'AWN (reproduced)', 'SAG-AWN (ours)'],
    'Overall Accuracy': ['~56.1%',        '~58.3%',         '64.32%',           '69.08%'],
    'Macro F1-Score':   ['—',             '—',              '66.14%',           '68.67%'],
    'Kappa':            ['—',             '—',              '0.6048',           '0.6569'],
})
print(results.to_string(index=False))
print('\n* CNN and LSTM values from O\'Shea & Hoydis (2017)')

---
## 7. Ablation Study

We validate that improvement comes from SNR-aware gating, not just extra parameters.

In [ ]:
!python ablation_study.py

In [ ]:
# Ablation results
ablation = pd.DataFrame({
    'Configuration':   [
        'AWN Baseline',
        'SAG-AWN (constant zero-SNR input)',
        'SAG-AWN (noisy SNR ±5%)',
        'SAG-AWN (oracle SNR)'
    ],
    'Accuracy': ['64.32%', '26.09%', '67.32%', '69.08%'],
    'vs Baseline': ['—', '−38.23%', '+3.00%', '+4.77%'],
})
print(ablation.to_string(index=False))
print('\nKey finding: Zeroing SNR input causes 38% drop → SAG genuinely uses SNR information.')

---
## 8. Training Curves

In [ ]:
!python plot_loss_curve.py
from IPython.display import Image
Image('./visuals/training_curves.png')

---
## 9. Save All Results to Drive

In [ ]:
import shutil

shutil.copytree('./visuals',
    '/content/drive/MyDrive/MYZ307E_AWN_Project/visuals',
    dirs_exist_ok=True)

shutil.copy('./README.md',
    '/content/drive/MyDrive/MYZ307E_AWN_Project/README.md')

print('✅ All results saved to Drive!')

---
## Summary

| Metric | Baseline AWN | SAG-AWN (Ours) | Improvement |
|--------|-------------|----------------|-------------|
| Overall Accuracy | 64.32% | **69.08%** | +4.77% |
| Macro F1-Score | 66.14% | **68.67%** | +2.53% |
| Kappa Coefficient | 0.6048 | **0.6569** | +0.0521 |

**Key Findings:**
- SAG improves accuracy across ALL SNR levels
- Largest gains in transition region: −12 to −6 dB (+11–13%)
- Zeroing SNR input causes 38% drop → SAG genuinely leverages SNR
- Robust to SNR estimation noise: noisy SNR still gives +3% improvement

**References:**  
[1] Zhang et al., IEEE TCCN 2023. DOI: 10.1109/TCCN.2023.3252580  
[2] O'Shea & Hoydis, IEEE TCCN 2017.